# 02 - Ingestão do PPI

## Objetivo
Realizar a ingestão do histórico de Preços de Paridade de Importação (PPI), disponibilizado pela Agência Nacional do Petróleo, Gás Natural e Biocombustíveis (ANP), para a camada Bronze do Lakehouse.

## Origem
Arquivo Excel armazenado no Volume RAW do Databricks.

## Destino
Tabelas:
- `workspace.bronze.ppi_gasolina_raw`
- `workspace.bronze.ppi_diesel_raw`

In [0]:
caminho_ppi = "/Volumes/workspace/raw/dados_raw/ppi/ppi_anp.xlsx.xlsx"

print(caminho_ppi)

/Volumes/workspace/raw/dados_raw/ppi/ppi_anp.xlsx.xlsx


## Leitura e inspeção do arquivo

O arquivo de PPI está em formato Excel (`.xlsx`) e contém séries semanais para diferentes produtos. A leitura é realizada com Pandas e `openpyxl`.

Antes da ingestão, são identificadas as abas disponíveis no arquivo. Para o escopo deste projeto, são utilizadas as séries de Gasolina e Diesel, que serão posteriormente relacionadas aos preços dos respectivos combustíveis ao consumidor.

In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd

xls_ppi = pd.ExcelFile(caminho_ppi)

print(xls_ppi.sheet_names)

['Gasolina R$ semanal', 'Diesel R$ semanal', 'QAV R$ semanal', 'GLP R$ kg semanal']


In [0]:
df_ppi_gasolina = pd.read_excel(
    caminho_ppi,
    sheet_name="Gasolina R$ semanal"
)

df_ppi_gasolina.head(15)

,Unnamed: 0,Gasolina A Comum,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Variação % em relação à semana anterior,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34
0,NaN,Preço de Paridade de Importação R$/litro,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Data,Manaus,Itaqui,Suape,Aratu,Santos,Paranagua,Tramandai,Guamaré,Duque de Caxias,Betim,Cubatão,Mauá,Paulínia,São José dos Campos,Araucária,Canoas,NaN,Manaus,Itaqui,Suape,Aratu,Santos,Paranagua,Tramandai,Guamaré,Duque de Caxias,Betim,Cubatão,Mauá,Paulínia,São José dos Campos,Araucária,Canoas
2,NaN,05/11/2018 a 09/11/2018,NaN,1.602604,1.616238,1.617774,1.653378,1.630454,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
3,NaN,12/11/2018 a 16/11/2018,NaN,1.524086,1.537714,1.539262,1.574876,1.551936,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-0.048994,-0.048584,-0.048531,-0.04748,-0.048157,-,-,-,-,-,-,-,-,-,-
4,NaN,19/11/2018 a 23/11/2018,NaN,1.48227,1.495897,1.497443,1.536817,1.51405,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-0.027437,-0.027194,-0.027168,-0.024167,-0.024412,-,-,-,-,-,-,-,-,-,-
5,NaN,26/11/2018 a 30/11/2018,NaN,1.40253,1.416134,1.417722,1.458436,1.435668,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-0.053796,-0.053321,-0.053238,-0.051002,-0.05177,-,-,-,-,-,-,-,-,-,-
6,NaN,03/12/2018 a 07/12/2018,NaN,1.4132,1.4268,1.4284,1.4754,1.4529,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,0.007608,0.007532,0.007532,0.011632,0.012003,-,-,-,-,-,-,-,-,-,-
7,NaN,10/12/2018 a 14/12/2018,NaN,1.4263,1.44,1.4415,1.4892,1.4667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,0.00927,0.009251,0.009171,0.009353,0.009498,-,-,-,-,-,-,-,-,-,-
8,NaN,17/12/2018 a 21/12/2018,NaN,1.354148,1.367754,1.369344,1.40937,1.386564,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-0.050587,-0.050171,-0.050056,-0.053606,-0.054637,-,-,-,-,-,-,-,-,-,-
9,NaN,24/12/2018 a 28/12/2018,NaN,1.272853,1.2957,1.303328,1.323986,1.304272,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-0.060034,-0.05268,-0.04821,-0.060583,-0.05935,-,-,-,-,-,-,-,-,-,-


### Identificação do cabeçalho da série

A inspeção inicial mostra que as duas primeiras linhas da planilha contêm informações auxiliares de cabeçalho e não correspondem às observações da série histórica.

Por esse motivo, a aba é relida com `skiprows=2`, fazendo com que a linha que contém os nomes dos campos seja utilizada como cabeçalho do DataFrame.

In [0]:
df_ppi_gasolina = pd.read_excel(
    caminho_ppi,
    sheet_name="Gasolina R$ semanal",
    skiprows=2
)

df_ppi_gasolina.head(10)

,Unnamed: 0,Data,Manaus,Itaqui,Suape,Aratu,Santos,Paranagua,Tramandai,Guamaré,Duque de Caxias,Betim,Cubatão,Mauá,Paulínia,São José dos Campos,Araucária,Canoas,Unnamed: 18,Manaus.1,Itaqui .1,Suape .1,Aratu .1,Santos .1,Paranagua .1,Tramandai.1,Guamaré.1,Duque de Caxias.1,Betim.1,Cubatão.1,Mauá.1,Paulínia.1,São José dos Campos.1,Araucária.1,Canoas.1
0,NaN,05/11/2018 a 09/11/2018,NaN,1.602604,1.616238,1.617774,1.653378,1.630454,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
1,NaN,12/11/2018 a 16/11/2018,NaN,1.524086,1.537714,1.539262,1.574876,1.551936,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-0.048994,-0.048584,-0.048531,-0.04748,-0.048157,-,-,-,-,-,-,-,-,-,-
2,NaN,19/11/2018 a 23/11/2018,NaN,1.482270,1.495897,1.497443,1.536817,1.514050,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-0.027437,-0.027194,-0.027168,-0.024167,-0.024412,-,-,-,-,-,-,-,-,-,-
3,NaN,26/11/2018 a 30/11/2018,NaN,1.402530,1.416134,1.417722,1.458436,1.435668,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-0.053796,-0.053321,-0.053238,-0.051002,-0.05177,-,-,-,-,-,-,-,-,-,-
4,NaN,03/12/2018 a 07/12/2018,NaN,1.413200,1.426800,1.428400,1.475400,1.452900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,0.007608,0.007532,0.007532,0.011632,0.012003,-,-,-,-,-,-,-,-,-,-
5,NaN,10/12/2018 a 14/12/2018,NaN,1.426300,1.440000,1.441500,1.489200,1.466700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,0.00927,0.009251,0.009171,0.009353,0.009498,-,-,-,-,-,-,-,-,-,-
6,NaN,17/12/2018 a 21/12/2018,NaN,1.354148,1.367754,1.369344,1.409370,1.386564,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-0.050587,-0.050171,-0.050056,-0.053606,-0.054637,-,-,-,-,-,-,-,-,-,-
7,NaN,24/12/2018 a 28/12/2018,NaN,1.272853,1.295700,1.303328,1.323986,1.304272,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-0.060034,-0.05268,-0.04821,-0.060583,-0.05935,-,-,-,-,-,-,-,-,-,-
8,NaN,31/12/2018 a 04/01/2019,NaN,1.262107,1.275827,1.277187,1.311913,1.289253,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-0.008442,-0.015338,-0.020057,-0.009119,-0.011515,-,-,-,-,-,-,-,-,-,-
9,NaN,07/01/2019 a 11/01/2019,NaN,1.291026,1.304758,1.306098,1.339580,1.316866,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,0.022914,0.022677,0.022637,0.021089,0.021418,-,-,-,-,-,-,-,-,-,-


In [0]:
df_ppi_diesel = pd.read_excel(
    caminho_ppi,
    sheet_name="Diesel R$ semanal",
    skiprows=2
)

df_ppi_diesel.head(10)

,Unnamed: 0,Data,Manaus,Itaqui,Suape,Aratu,Santos,Paranagua,Tramandai,Guamaré,Duque de Caxias,Betim,Cubatão,Mauá,Paulínia,São José dos Campos,Araucária,Canoas,Unnamed: 18,Manaus.1,Itaqui .1,Suape .1,Aratu .1,Santos .1,Paranagua .1,Tramandai.1,Guamaré.1,Duque de Caxias.1,Betim.1,Cubatão.1,Mauá.1,Paulínia.1,São José dos Campos.1,Araucária.1,Canoas.1
0,NaN,05/11/2018 a 09/11/2018,NaN,2.249686,2.250400,2.264426,2.299988,2.281410,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
1,NaN,12/11/2018 a 16/11/2018,NaN,2.159192,2.160458,2.174492,2.210074,2.191472,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-0.040225,-0.039967,-0.039716,-0.039093,-0.039422,-,-,-,-,-,-,-,-,-,-
2,NaN,17/11/2018 a 21/11/2018,NaN,2.103863,2.102917,2.119167,2.158400,2.140837,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-0.025625,-0.026634,-0.025443,-0.023381,-0.023106,-,-,-,-,-,-,-,-,-,-
3,NaN,26/11/2018 a 30/11/2018,NaN,1.990058,1.988348,2.005388,2.045942,2.028652,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-0.054094,-0.054481,-0.05369,-0.052102,-0.052402,-,-,-,-,-,-,-,-,-,-
4,NaN,03/12/2018 a 07/12/2018,NaN,1.993900,1.988500,2.009200,2.055900,2.040400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,0.001931,0.000076,0.001901,0.004867,0.005791,-,-,-,-,-,-,-,-,-,-
5,NaN,10/12/2018 a 14/12/2018,NaN,1.981700,1.975900,1.997000,2.044300,2.028900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-0.006119,-0.006336,-0.006072,-0.005642,-0.005636,-,-,-,-,-,-,-,-,-,-
6,NaN,17/12/2018 a 21/12/2018,NaN,1.874776,1.873470,1.890108,1.929994,1.912502,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-0.053956,-0.05184,-0.053526,-0.055914,-0.05737,-,-,-,-,-,-,-,-,-,-
7,NaN,24/12/2018 a 28/12/2018,NaN,1.746813,1.769660,1.777288,1.797946,1.778232,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-0.068255,-0.05541,-0.05969,-0.068419,-0.070207,-,-,-,-,-,-,-,-,-,-
8,NaN,31/12/2018 a 04/01/2019,NaN,1.719510,1.721637,1.734710,1.770437,1.752157,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,-0.01563,-0.027137,-0.023957,-0.015301,-0.014664,-,-,-,-,-,-,-,-,-,-
9,NaN,07/01/2019 a 11/01/2019,NaN,1.811598,1.814008,1.826780,1.860204,1.841488,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,0.053555,0.053653,0.053075,0.050703,0.050984,-,-,-,-,-,-,-,-,-,-


## Verificação inicial dos dados

Após a leitura das séries de Gasolina e Diesel, é realizada uma verificação inicial das dimensões, tipos de dados e valores nulos.

Essa etapa permite identificar características da estrutura original que precisam ser consideradas antes da conversão para Spark e da persistência na camada Bronze.

In [0]:
print("GASOLINA")
print(f"Linhas: {df_ppi_gasolina.shape[0]}")
print(f"Colunas: {df_ppi_gasolina.shape[1]}")
print("\nNulos por coluna:")
print(df_ppi_gasolina.isnull().sum())

print("\n" + "="*50)

print("DIESEL")
print(f"Linhas: {df_ppi_diesel.shape[0]}")
print(f"Colunas: {df_ppi_diesel.shape[1]}")
print("\nNulos por coluna:")
print(df_ppi_diesel.isnull().sum())

GASOLINA
Linhas: 409
Colunas: 35

Nulos por coluna:
Unnamed: 0               409
Data                       0
Manaus                    43
Itaqui                     0
Suape                      0
Aratu                      0
Santos                     0
Paranagua                  0
Tramandai                 43
Guamaré                   43
Duque de Caxias           43
Betim                     43
Cubatão                   43
Mauá                      43
Paulínia                  43
São José dos Campos       43
Araucária                 43
Canoas                    43
Unnamed: 18              409
Manaus.1                   0
Itaqui         .1          0
Suape         .1           0
Aratu         .1           0
Santos     .1              0
Paranagua       .1         0
Tramandai.1                0
Guamaré.1                  0
Duque de Caxias.1          0
Betim.1                    0
Cubatão.1                  0
Mauá.1                     0
Paulínia.1                 0
São José dos Campos.

## Padronização da estrutura

A inspeção do arquivo identificou dois blocos de informações para cada localidade: o **Preço de Paridade de Importação (PPI)** e sua **variação percentual em relação à semana anterior**.

Para tornar o schema mais claro e adequado ao processamento no Spark:

- Os nomes das colunas são normalizados, com remoção de acentos e caracteres especiais;
- As colunas do segundo bloco recebem o sufixo `_variacao_pct`;
- Colunas auxiliares do arquivo Excel, sem informação analítica, são removidas;
- Os valores de PPI e de variação são convertidos para tipo numérico, sendo valores não numéricos da fonte convertidos para nulos.

O arquivo original permanece preservado na camada RAW.

In [0]:
import re
import unicodedata

def normalizar_nome_coluna(nome):
    # Remove acentos
    nome = unicodedata.normalize("NFKD", str(nome))
    nome = nome.encode("ascii", "ignore").decode("ascii")

    # Substitui caracteres especiais por _
    nome = re.sub(r"[^a-zA-Z0-9_]", "_", nome)

    # Remove _ repetidos
    nome = re.sub(r"_+", "_", nome)

    # Remove _ no início/fim
    return nome.strip("_")

In [0]:
print("GASOLINA")

for original in df_ppi_gasolina.columns:
    print(f"{original}  -->  {normalizar_nome_coluna(original)}")

GASOLINA
Unnamed: 0  -->  Unnamed_0
Data  -->  Data
Manaus  -->  Manaus
Itaqui           -->  Itaqui
Suape           -->  Suape
Aratu           -->  Aratu
Santos       -->  Santos
Paranagua         -->  Paranagua
Tramandai  -->  Tramandai
Guamaré  -->  Guamare
Duque de Caxias  -->  Duque_de_Caxias
Betim  -->  Betim
Cubatão  -->  Cubatao
Mauá  -->  Maua
Paulínia  -->  Paulinia
São José dos Campos  -->  Sao_Jose_dos_Campos
Araucária  -->  Araucaria
Canoas  -->  Canoas
Unnamed: 18  -->  Unnamed_18
Manaus.1  -->  Manaus_1
Itaqui         .1  -->  Itaqui_1
Suape         .1  -->  Suape_1
Aratu         .1  -->  Aratu_1
Santos     .1  -->  Santos_1
Paranagua       .1  -->  Paranagua_1
Tramandai.1  -->  Tramandai_1
Guamaré.1  -->  Guamare_1
Duque de Caxias.1  -->  Duque_de_Caxias_1
Betim.1  -->  Betim_1
Cubatão.1  -->  Cubatao_1
Mauá.1  -->  Maua_1
Paulínia.1  -->  Paulinia_1
São José dos Campos.1  -->  Sao_Jose_dos_Campos_1
Araucária.1  -->  Araucaria_1
Canoas.1  -->  Canoas_1


In [0]:
novos_nomes_gasolina = [
    normalizar_nome_coluna(col)
    for col in df_ppi_gasolina.columns
]

print(f"Total de colunas: {len(novos_nomes_gasolina)}")
print(f"Nomes únicos: {len(set(novos_nomes_gasolina))}")

if len(novos_nomes_gasolina) == len(set(novos_nomes_gasolina)):
    print("OK - Todos os nomes de colunas são únicos.")
else:
    print("ATENÇÃO - Existem nomes duplicados após a normalização.")

Total de colunas: 35
Nomes únicos: 35
OK - Todos os nomes de colunas são únicos.


In [0]:
def renomear_colunas_ppi(df):
    novas_colunas = {}

    for coluna in df.columns:
        nome_normalizado = normalizar_nome_coluna(coluna)

        # O sufixo .1 identifica o segundo bloco de localidades,
        # correspondente à variação percentual semanal
        if coluna.endswith(".1"):
            nome_base = normalizar_nome_coluna(coluna[:-2])
            nome_normalizado = f"{nome_base}_variacao_pct"

        novas_colunas[coluna] = nome_normalizado

    return df.rename(columns=novas_colunas)


df_ppi_gasolina = renomear_colunas_ppi(df_ppi_gasolina)
df_ppi_diesel = renomear_colunas_ppi(df_ppi_diesel)

colunas_auxiliares = ["Unnamed_0", "Unnamed_18"]

df_ppi_gasolina = df_ppi_gasolina.drop(
    columns=colunas_auxiliares,
    errors="ignore"
)

df_ppi_diesel = df_ppi_diesel.drop(
    columns=colunas_auxiliares,
    errors="ignore"
)

print("Colunas da Gasolina:")
print(df_ppi_gasolina.columns.tolist())

print("\nColunas do Diesel:")
print(df_ppi_diesel.columns.tolist())

Colunas da Gasolina:
['Data', 'Manaus', 'Itaqui', 'Suape', 'Aratu', 'Santos', 'Paranagua', 'Tramandai', 'Guamare', 'Duque_de_Caxias', 'Betim', 'Cubatao', 'Maua', 'Paulinia', 'Sao_Jose_dos_Campos', 'Araucaria', 'Canoas', 'Manaus_variacao_pct', 'Itaqui_variacao_pct', 'Suape_variacao_pct', 'Aratu_variacao_pct', 'Santos_variacao_pct', 'Paranagua_variacao_pct', 'Tramandai_variacao_pct', 'Guamare_variacao_pct', 'Duque_de_Caxias_variacao_pct', 'Betim_variacao_pct', 'Cubatao_variacao_pct', 'Maua_variacao_pct', 'Paulinia_variacao_pct', 'Sao_Jose_dos_Campos_variacao_pct', 'Araucaria_variacao_pct', 'Canoas_variacao_pct']

Colunas do Diesel:
['Data', 'Manaus', 'Itaqui', 'Suape', 'Aratu', 'Santos', 'Paranagua', 'Tramandai', 'Guamare', 'Duque_de_Caxias', 'Betim', 'Cubatao', 'Maua', 'Paulinia', 'Sao_Jose_dos_Campos', 'Araucaria', 'Canoas', 'Manaus_variacao_pct', 'Itaqui_variacao_pct', 'Suape_variacao_pct', 'Aratu_variacao_pct', 'Santos_variacao_pct', 'Paranagua_variacao_pct', 'Tramandai_variacao_pct'

In [0]:
# Converte as colunas de valores e variações para tipo numérico.
# Valores não numéricos da fonte, como "-", são convertidos para NaN.

colunas_numericas_gasolina = [
    col for col in df_ppi_gasolina.columns
    if col != "Data"
]

colunas_numericas_diesel = [
    col for col in df_ppi_diesel.columns
    if col != "Data"
]

df_ppi_gasolina[colunas_numericas_gasolina] = (
    df_ppi_gasolina[colunas_numericas_gasolina]
    .apply(pd.to_numeric, errors="coerce")
)

df_ppi_diesel[colunas_numericas_diesel] = (
    df_ppi_diesel[colunas_numericas_diesel]
    .apply(pd.to_numeric, errors="coerce")
)

In [0]:
df_ppi_gasolina_spark = spark.createDataFrame(df_ppi_gasolina)
df_ppi_diesel_spark = spark.createDataFrame(df_ppi_diesel)

print("Schema Gasolina:")
df_ppi_gasolina_spark.printSchema()

print("\nSchema Diesel:")
df_ppi_diesel_spark.printSchema()

Schema Gasolina:
root
 |-- Data: string (nullable = true)
 |-- Manaus: double (nullable = true)
 |-- Itaqui: double (nullable = true)
 |-- Suape: double (nullable = true)
 |-- Aratu: double (nullable = true)
 |-- Santos: double (nullable = true)
 |-- Paranagua: double (nullable = true)
 |-- Tramandai: double (nullable = true)
 |-- Guamare: double (nullable = true)
 |-- Duque_de_Caxias: double (nullable = true)
 |-- Betim: double (nullable = true)
 |-- Cubatao: double (nullable = true)
 |-- Maua: double (nullable = true)
 |-- Paulinia: double (nullable = true)
 |-- Sao_Jose_dos_Campos: double (nullable = true)
 |-- Araucaria: double (nullable = true)
 |-- Canoas: double (nullable = true)
 |-- Manaus_variacao_pct: double (nullable = true)
 |-- Itaqui_variacao_pct: double (nullable = true)
 |-- Suape_variacao_pct: double (nullable = true)
 |-- Aratu_variacao_pct: double (nullable = true)
 |-- Santos_variacao_pct: double (nullable = true)
 |-- Paranagua_variacao_pct: double (nullable = tru

## Persistência na camada Bronze

As séries de Gasolina e Diesel são persistidas separadamente em formato Delta nas tabelas da camada Bronze.

A gravação utiliza substituição controlada do schema para permitir a recriação das tabelas com a estrutura definida neste processo de ingestão.

Após a persistência, as tabelas são novamente carregadas e suas quantidades de registros são comparadas às dos DataFrames de origem, verificando se houve perda de registros durante a gravação.

In [0]:
(
    df_ppi_gasolina_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.ppi_gasolina_raw")
)

(
    df_ppi_diesel_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.ppi_diesel_raw")
)

In [0]:
gasolina_bronze = spark.table("workspace.bronze.ppi_gasolina_raw")
diesel_bronze = spark.table("workspace.bronze.ppi_diesel_raw")

print(f"Gasolina origem: {df_ppi_gasolina_spark.count()}")
print(f"Gasolina Bronze: {gasolina_bronze.count()}")

print(f"Diesel origem: {df_ppi_diesel_spark.count()}")
print(f"Diesel Bronze: {diesel_bronze.count()}")

Gasolina origem: 409
Gasolina Bronze: 409
Diesel origem: 409
Diesel Bronze: 409
